In [25]:
import pandas as pd
import numpy as np
import tensorflow as tf

train_df = pd.read_parquet(r"train_df_128.parquet")
val_df   = pd.read_parquet(r"val_df_128.parquet")

# overview_cols needed for ITEM_FEATS definition later
overview_cols = [c for c in train_df.columns if c.startswith('overview_')]

print(f"Train rows: {len(train_df):,} | Val rows: {len(val_df):,}")

Train rows: 964,331 | Val rows: 173,382


In [26]:
all_df = pd.concat([train_df, val_df], ignore_index=True)

user_id_map  = {uid: idx for idx, uid in enumerate(sorted(all_df['userId'].unique()))}
movie_id_map = {mid: idx for idx, mid in enumerate(sorted(all_df['movieId'].unique()))}
idx_to_movie = {v: k for k, v in movie_id_map.items()}

# Remap indices using the new consistent maps
train_df['user_idx']  = train_df['userId'].map(user_id_map)
train_df['movie_idx'] = train_df['movieId'].map(movie_id_map)
val_df['user_idx']    = val_df['userId'].map(user_id_map)
val_df['movie_idx']   = val_df['movieId'].map(movie_id_map)

n_users  = len(user_id_map)
n_movies = len(movie_id_map)

affinity_cols = [c for c in train_df.columns if c.startswith('affinity_')]
genre_cols    = [c for c in train_df.columns if c.startswith('genre_')]

USER_FEATS = [
    'user_mean_rating', 'user_rating_std',
    'user_rating_count', 'rating_deviation'
] + affinity_cols

ITEM_FEATS = [
    'cf_mean_rating', 'cf_rating_std', 'log_cf_count',
    'log_popularity', 'log_vote_count', 'release_year',
    'runtime', 'vote_average', 'in_collection', 'collection_size'
] + genre_cols + overview_cols

# ITEM_FEATS = [
#     'cf_mean_rating', 'cf_rating_std', 'log_cf_count',
#     'log_popularity', 'log_vote_count', 'release_year',
#     'runtime', 'vote_average', 'in_collection', 'collection_size'
# ] + genre_cols
# No overview_cols

missing_user = [c for c in USER_FEATS if c not in train_df.columns]
missing_item = [c for c in ITEM_FEATS if c not in train_df.columns]
print("Missing user feats:", missing_user)
print("Missing item feats:", missing_item)

train_df[USER_FEATS + ITEM_FEATS] = train_df[USER_FEATS + ITEM_FEATS].fillna(0)
val_df[USER_FEATS + ITEM_FEATS]   = val_df[USER_FEATS + ITEM_FEATS].fillna(0)

user_feat_dim = len(USER_FEATS)
item_feat_dim = len(ITEM_FEATS)

print(f"Users: {n_users:,} | Movies: {n_movies:,}")
print(f"User feat dim: {user_feat_dim} | Item feat dim: {item_feat_dim}")

Missing user feats: []
Missing item feats: []
Users: 50,000 | Movies: 4,991
User feat dim: 4 | Item feat dim: 158


In [27]:
# Data already split and scaled in preprocessing
assert train_df[USER_FEATS + ITEM_FEATS].isnull().sum().sum() == 0
assert val_df[USER_FEATS + ITEM_FEATS].isnull().sum().sum() == 0

print(f"Train rows: {len(train_df):,} | Val rows: {len(val_df):,}")
print(f"User feat dim: {user_feat_dim} | Item feat dim: {item_feat_dim}")

Train rows: 964,331 | Val rows: 173,382
User feat dim: 4 | Item feat dim: 158


In [28]:
# Make sure user_idx and movie_idx are columns not index
assert 'user_idx' in train_df.columns, "user_idx missing from columns!"
assert 'movie_idx' in train_df.columns, "movie_idx missing from columns!"
print(train_df[['user_idx', 'movie_idx']].head())

   user_idx  movie_idx
0         0        233
1         0        400
2         0        553
3         0        652
4         0        808


In [29]:
BATCH_SIZE = 4096
AUTOTUNE   = tf.data.AUTOTUNE

def make_dataset(df, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices({
        'user_id':    df['user_idx'].values.astype(np.int32),
        'user_feats': df[USER_FEATS].values.astype(np.float32),
        'item_id':    df['movie_idx'].values.astype(np.int32),
        'item_feats': df[ITEM_FEATS].values.astype(np.float32),
    })
    if shuffle:
        ds = ds.shuffle(len(df), seed=None)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True)
val_ds   = make_dataset(val_df,   shuffle=False)

In [ ]:
EMB_DIM = 128

def build_user_tower():
    user_id_input    = tf.keras.Input(shape=(), name='user_id', dtype='int32')
    user_feats_input = tf.keras.Input(shape=(user_feat_dim,), name='user_feats')

    u = tf.keras.layers.Embedding(n_users, EMB_DIM, name='user_embedding',
                                   embeddings_initializer='he_normal')(user_id_input)
    u = tf.keras.layers.Flatten()(u)

    # Project user features separately before concat
    f = tf.keras.layers.Dense(64, activation='relu')(user_feats_input)
    f = tf.keras.layers.LayerNormalization()(f)

    u = tf.keras.layers.Concatenate()([u, f])
    u = tf.keras.layers.Dense(128, kernel_regularizer=tf.keras.regularizers.l2(1e-4))(u)
    u = tf.keras.layers.LayerNormalization()(u)
    u = tf.keras.layers.ReLU()(u)
    u = tf.keras.layers.Dropout(0.4)(u)
    u = tf.keras.layers.Dense(64, kernel_regularizer=tf.keras.regularizers.l2(1e-4))(u)
    u = tf.keras.layers.LayerNormalization()(u)
    u = tf.keras.layers.ReLU()(u)
    u = tf.keras.layers.Dense(EMB_DIM)(u)
    u = tf.keras.layers.LayerNormalization()(u)
    u = tf.keras.layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))(u)

    return tf.keras.Model([user_id_input, user_feats_input], u, name='user_tower')

# def build_item_tower(n_movies, item_feat_dim, emb_dim=128):
#     item_id_input    = tf.keras.Input(shape=(), name='item_id', dtype='int32')
#     item_feats_input = tf.keras.Input(shape=(item_feat_dim,), name='item_feats')

#     i = tf.keras.layers.Embedding(n_movies, emb_dim, name='item_embedding',
#                                    embeddings_initializer='he_normal')(item_id_input)
#     i = tf.keras.layers.Flatten()(i)
#     i = tf.keras.layers.Concatenate()([i, item_feats_input])
#     i = tf.keras.layers.Dense(256, kernel_regularizer=tf.keras.regularizers.l2(1e-4))(i)
#     i = tf.keras.layers.LayerNormalization()(i)
#     i = tf.keras.layers.ReLU()(i)
#     i = tf.keras.layers.Dropout(0.2)(i)
#     i = tf.keras.layers.Dense(128, kernel_regularizer=tf.keras.regularizers.l2(1e-4))(i)
#     i = tf.keras.layers.LayerNormalization()(i)
#     i = tf.keras.layers.ReLU()(i)
#     i = tf.keras.layers.Dense(emb_dim)(i)
#     i = tf.keras.layers.LayerNormalization()(i)
#     i = tf.keras.layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))(i)

#     return tf.keras.Model([item_id_input, item_feats_input], i, name='item_tower')
n_overview = 128
n_other    = item_feat_dim - n_overview  # 30

def build_item_tower(n_movies, item_feat_dim, emb_dim=128):
    item_id_input    = tf.keras.Input(shape=(), name='item_id', dtype='int32')
    item_feats_input = tf.keras.Input(shape=(item_feat_dim,), name='item_feats')

    i = tf.keras.layers.Embedding(n_movies, emb_dim, name='item_embedding',
                                   embeddings_initializer='he_normal')(item_id_input)
    i = tf.keras.layers.Flatten()(i)

    other_feats    = tf.keras.layers.Lambda(lambda x: x[:, :n_other])(item_feats_input)
    overview_feats = tf.keras.layers.Lambda(lambda x: x[:, n_other:])(item_feats_input)
    overview_proj  = tf.keras.layers.Dense(32, activation='relu')(overview_feats)

    i = tf.keras.layers.Concatenate()([i, other_feats, overview_proj])
    i = tf.keras.layers.Dense(128, kernel_regularizer=tf.keras.regularizers.l2(1e-4))(i)
    i = tf.keras.layers.LayerNormalization()(i)
    i = tf.keras.layers.ReLU()(i)
    i = tf.keras.layers.Dropout(0.4)(i)
    i = tf.keras.layers.Dense(64, kernel_regularizer=tf.keras.regularizers.l2(1e-4))(i)
    i = tf.keras.layers.LayerNormalization()(i)
    i = tf.keras.layers.ReLU()(i)
    i = tf.keras.layers.Dense(emb_dim)(i)
    i = tf.keras.layers.LayerNormalization()(i)
    i = tf.keras.layers.Lambda(lambda x: tf.math.l2_normalize(x, axis=1))(i)

    return tf.keras.Model([item_id_input, item_feats_input], i, name='item_tower')

user_model = build_user_tower()
item_model = build_item_tower(n_movies=n_movies, item_feat_dim=item_feat_dim, emb_dim=EMB_DIM)

In [31]:
class TwoTowerModel(tf.keras.Model):
    def __init__(self, user_model, item_model, init_temperature=0.07):
        super().__init__()
        self.user_model  = user_model
        self.item_model  = item_model
        self.log_temp    = tf.Variable(np.log(init_temperature), dtype=tf.float32, trainable=True)

    @property
    def temperature(self):
        return tf.exp(self.log_temp)

    def call(self, inputs, training=False):  # ← was missing
        u = self.user_model([inputs['user_id'], inputs['user_feats']], training=training)
        i = self.item_model([inputs['item_id'], inputs['item_feats']], training=training)
        return u, i

    def compute_loss(self, u_emb, i_emb):
        u_emb  = tf.math.l2_normalize(u_emb, axis=1)
        i_emb  = tf.math.l2_normalize(i_emb, axis=1)
        logits = tf.matmul(u_emb, i_emb, transpose_b=True) / self.temperature
        batch_size = tf.shape(logits)[0]
        labels = tf.range(batch_size)
        eye    = tf.eye(batch_size)
        false_neg_mask = tf.cast(logits > 0.8, tf.float32) * (1.0 - eye)
        logits = logits - false_neg_mask * 1e9
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
        return (loss_fn(labels, logits) + loss_fn(labels, tf.transpose(logits))) / 2.0

    def train_step(self, data):
        with tf.GradientTape() as tape:
            u_emb, i_emb = self(data, training=True)
            loss = self.compute_loss(u_emb, i_emb)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return {'loss': loss, 'recall@10': self.recall_at_k(u_emb, i_emb, k=10)}

    def test_step(self, data):
        u_emb, i_emb = self(data, training=False)
        loss = self.compute_loss(u_emb, i_emb)
        return {'loss': loss, 'recall@10': self.recall_at_k(u_emb, i_emb, k=10)}

    def recall_at_k(self, u_emb, i_emb, k=10):
        u_emb  = tf.math.l2_normalize(u_emb, axis=1)
        i_emb  = tf.math.l2_normalize(i_emb, axis=1)
        logits = tf.matmul(u_emb, i_emb, transpose_b=True)
        batch_size = tf.shape(u_emb)[0]
        k_eff  = tf.minimum(k, batch_size)
        top_k  = tf.math.top_k(logits, k=k_eff).indices
        targets = tf.expand_dims(tf.range(batch_size), axis=1)
        hits   = tf.reduce_any(top_k == targets, axis=1)
        return tf.reduce_mean(tf.cast(hits, tf.float32))

In [32]:
two_tower = TwoTowerModel(user_model, item_model)
two_tower.compile(optimizer=tf.keras.optimizers.Adam(1e-3))

history = two_tower.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=7,
            restore_best_weights=True,
            monitor='val_loss'
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            patience=3,
            factor=0.5,
            monitor='val_loss',
            min_delta=0.001
        )
    ]
)

Epoch 1/40


236/236 ━━━━━━━━━━━━━━━━━━━━ 147s 595ms/step - loss: 5.9415 - recall@10: 0.0113 - val_loss: 6.7481 - val_recall@10: 0.0074 - learning_rate: 0.0010
Epoch 2/40
236/236 ━━━━━━━━━━━━━━━━━━━━ 1629s 7s/step - loss: 5.4849 - recall@10: 0.0119 - val_loss: 6.7426 - val_recall@10: 0.0089 - learning_rate: 0.0010
Epoch 3/40
236/236 ━━━━━━━━━━━━━━━━━━━━ 128s 533ms/step - loss: 5.1158 - recall@10: 0.0209 - val_loss: 6.6973 - val_recall@10: 0.0074 - learning_rate: 0.0010
Epoch 4/40
236/236 ━━━━━━━━━━━━━━━━━━━━ 1065s 5s/step - loss: 4.5079 - recall@10: 0.0209 - val_loss: 6.7500 - val_recall@10: 0.0096 - learning_rate: 0.0010
Epoch 5/40
236/236 ━━━━━━━━━━━━━━━━━━━━ 200s 831ms/step - loss: 4.5188 - recall@10: 0.0198 - val_loss: 6.7807 - val_recall@10: 0.0074 - learning_rate: 0.0010
Epoch 6/40
236/236 ━━━━━━━━━━━━━━━━━━━━ 189s 785ms/step - loss: 4.1777 - recall@10: 0.0265 - val_loss: 6.7042 - val_recall@10: 0.0074 - learning_rate: 0.0010
Epoch 7/40
236/236 ━━━━━━━━━━━━━━━━━━━━ 190s 794ms/step - loss: 4.0

In [33]:
movie_lookup = (
    train_df[['movie_idx', 'movieId'] + ITEM_FEATS]
    .drop_duplicates(subset='movie_idx')
    .sort_values('movie_idx')
    .reset_index(drop=True)
)

all_movie_idxs = movie_lookup['movie_idx'].values.astype(np.int32)
item_feats_all = movie_lookup[ITEM_FEATS].values.astype(np.float32)

# correct mapping (for decoding predictions)
movieid_from_idx = dict(zip(movie_lookup['movie_idx'], movie_lookup['movieId']))

# validation users
val_user_ids = val_df['userId'].drop_duplicates().values

# ground truth (relevant items in validation set)
val_user_pos = (
    val_df[val_df['rating'] >= 4.0]
    .groupby('userId')['movieId']
    .apply(set)
    .to_dict()
)

# IMPORTANT: only use TRAIN for "seen items" (avoid leakage)
val_user_seen = (
    train_df.groupby('userId')['movieId']
    .apply(set)
    .to_dict()
)

all_item_embs = item_model.predict(
    [all_movie_idxs, item_feats_all],
    batch_size=512,
    verbose=1
)

print('all_item_embs:', all_item_embs.shape)

print('movie_lookup:', movie_lookup.shape)
print('val users:', len(val_user_ids))

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
all_item_embs: (4878, 128)
movie_lookup: (4878, 160)
val users: 7500


EVALUATON

In [34]:
user_feat_map = (
    train_df.sort_values('timestamp')
    .groupby('user_idx')[USER_FEATS]
    .last()
)

item_pop = train_df.groupby('movie_idx').size()
item_pop = item_pop.reindex(all_movie_idxs).fillna(0)
item_pop = np.log1p(item_pop.values).astype(np.float32)

In [35]:
import numpy as np
from tqdm import tqdm

def evaluate_model(
    user_model, all_item_embs, train_df, val_df,
    user_feat_map, item_pop, all_movie_idxs,
    k=10, n_users=500, seed=42
):
    np.random.seed(seed)

    user_pos         = val_df.groupby('user_idx')['movie_idx'].apply(set).to_dict()
    user_seen_train  = train_df.groupby('user_idx')['movie_idx'].apply(set).to_dict()

    users         = list(user_pos.keys())
    sampled_users = np.random.choice(users, min(n_users, len(users)), replace=False)

    movie_idx_to_row = {idx: i for i, idx in enumerate(all_movie_idxs)}
    row_to_movie_idx = {i: idx for i, idx in enumerate(all_movie_idxs)}

    pop_bias = (item_pop - item_pop.mean()) / (item_pop.std() + 1e-8)

    # Fix 9: mean features for cold-start fallback
    mean_user_feats = user_feat_map.mean().values.astype(np.float32)

    recall_scores, hit_scores, ndcg_scores = [], [], []

    for user_id in tqdm(sampled_users):

        # Fix 9: fallback instead of skip
        if user_id not in user_feat_map.index:
            user_feats = mean_user_feats
        else:
            user_feats = user_feat_map.loc[user_id].values.astype(np.float32)

        u_emb = user_model.predict(
            [np.array([user_id], dtype=np.int32), user_feats.reshape(1, -1)],
            verbose=0
        ).reshape(-1)

        scores = all_item_embs @ u_emb
        scores = (scores - scores.mean()) / (scores.std() + 1e-8)
        scores += 0.05 * pop_bias

        seen_rows        = [movie_idx_to_row[i] for i in user_seen_train.get(user_id, set()) if i in movie_idx_to_row]
        scores[seen_rows] = -np.inf

        k_eff    = min(k, len(scores))
        top_k    = np.argpartition(-scores, k_eff)[:k_eff]
        top_k    = top_k[np.argsort(-scores[top_k])]
        top_k_movies = [row_to_movie_idx[i] for i in top_k]

        truth = user_pos[user_id]
        if len(truth) == 0:
            continue

        hits = len(set(top_k_movies) & truth)
        recall_scores.append(hits / len(truth))
        hit_scores.append(int(hits > 0))

        dcg  = sum(1 / np.log2(i + 2) for i, p in enumerate(top_k_movies) if p in truth)
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(truth), k)))
        ndcg_scores.append(dcg / idcg if idcg > 0 else 0)

    print(f"\n===== Evaluation @ {k} =====")
    if len(recall_scores) == 0:
        print("No valid users evaluated.")
        return
    print(f"Users evaluated: {len(recall_scores)}")
    print(f"Recall@{k}:  {np.mean(recall_scores):.4f}")
    print(f"HitRate@{k}: {np.mean(hit_scores):.4f}")
    print(f"NDCG@{k}:    {np.mean(ndcg_scores):.4f}")

In [36]:
for k in [1, 5, 10, 20, 50]:
    evaluate_model(
        user_model=user_model,
        all_item_embs=all_item_embs,
        train_df=train_df,
        val_df=val_df,
        user_feat_map=user_feat_map,
        item_pop=item_pop,
        all_movie_idxs=all_movie_idxs,
        k=k,
        n_users=500,
        seed=42
    )

100%|██████████| 500/500 [00:54<00:00,  9.23it/s]



===== Evaluation @ 1 =====
Users evaluated: 500
Recall@1:  0.0151
HitRate@1: 0.2200
NDCG@1:    0.2200


100%|██████████| 500/500 [00:49<00:00, 10.17it/s]



===== Evaluation @ 5 =====
Users evaluated: 500
Recall@5:  0.0640
HitRate@5: 0.4660
NDCG@5:    0.1920


100%|██████████| 500/500 [00:45<00:00, 11.02it/s]



===== Evaluation @ 10 =====
Users evaluated: 500
Recall@10:  0.1156
HitRate@10: 0.6040
NDCG@10:    0.1963


100%|██████████| 500/500 [00:49<00:00, 10.11it/s]



===== Evaluation @ 20 =====
Users evaluated: 500
Recall@20:  0.1611
HitRate@20: 0.6420
NDCG@20:    0.1881


100%|██████████| 500/500 [00:55<00:00,  9.06it/s]


===== Evaluation @ 50 =====
Users evaluated: 500
Recall@50:  0.1707
HitRate@50: 0.6500
NDCG@50:    0.1616
